# 03 · Visualising the comparison

**Mission covered:** 6

**Question:** how can we show at a glance what notebook 02 demonstrated with numbers?

**What we learn:** the structure of a matplotlib figure (`fig`, `ax`), choosing the right chart,
exporting images for the README.

**Rules of a good chart:** a title, labelled axes **with their unit**, a legend, distinct colours.

**How to use this notebook:** each step starts with a ✏️ **Exercise**. Try to answer it yourself
(add a new cell below it) before reading the ✅ **Solution** and the 💡 **Explanation** that follow.

## Setup

These cells are the same as in `01_exploration.ipynb`: imports, FastF1 cache and the `show` function.
The exercises and explanations about them are in notebook 01.

> 🔁 Copying this block into every notebook is not ideal: if it is fixed in one place, it has to be fixed everywhere.
> This is exactly the problem that **mission 8** will solve (moving this code into `src/`).

In [ ]:
import os

import fastf1
import fastf1.plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# The relative path "../cache" is only correct if the notebook runs from notebooks/
assert os.path.basename(os.getcwd()) == "notebooks", f"Unexpected working directory: {os.getcwd()}"

cache_dir = os.path.abspath("../cache")
os.makedirs(cache_dir, exist_ok=True)
print("FastF1 cache:", cache_dir)

fastf1.Cache.configure(cache_dir=cache_dir)

In [ ]:
def show(df, columns=None, n=None):
    '''Display a DataFrame as a Jupyter table, without truncated columns.

    columns : list of column names to keep (all by default)
    n       : number of rows to display (all by default)
    '''
    if columns is not None:
        df = df[columns]
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None):
        display(df)

### Loading the session


Same race as in `02_driver_comparison.ipynb`.

In [ ]:
YEAR = 2024
GRAND_PRIX = "Monaco"
SESSION_TYPE = "R"

session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION_TYPE)
session.load()
print(session.event["EventName"], "-", session.name)

In [ ]:
# Without a Formula 1 account sign-in, FastF1 only retrieves the results:
# accessing session.laps then raises a DataNotLoadedError.
try:
    laps_count = len(session.laps)
except fastf1.exceptions.DataNotLoadedError:
    laps_count = 0

if laps_count == 0:
    print("⚠️ No laps loaded. Sign in from a terminal with:")
    print("   python -m fastf1 auth f1tv --authenticate")
else:
    print(f"✅ {laps_count} laps loaded")

## 6.1 · Getting the clean laps

> ✏️ **Exercise**
>
> Reuse the driver choice, the filtering and the cleaning from notebook 02.
> (Yes, this is copied code: it will be factored out in mission 8.)

In [ ]:
# ✅ Solution
DRIVER_1 = "LEC"
DRIVER_2 = "PIA"
DRIVERS = [DRIVER_1, DRIVER_2]

In [ ]:
# ✅ Solution
laps = session.laps.pick_drivers(DRIVERS)

clean_laps = (
    laps
    .pick_wo_box()            # remove pit out-laps and in-laps
    .pick_track_status("1")   # keep only laps fully under green flag
    .pick_accurate()          # keep lap times FastF1 considers reliable
    .pick_not_deleted()       # remove laps deleted by race control (track limits)
)
clean_laps = clean_laps[clean_laps["LapNumber"] > 1]   # lap 1 (standing start) is not representative

# Lap time in seconds: a float is easier to work with than a duration
clean_laps = clean_laps.assign(LapTime_s=clean_laps["LapTime"].dt.total_seconds())

print("Representative laps:", clean_laps["Driver"].value_counts().to_dict())

## 6.2 · Preparing the export

> ✏️ **Exercise**
>
> Images go into `image/` at the root of the repo. Same logic as for the cache:
> from `notebooks/`, which relative path should you use? Store it in a single variable and make sure the folder exists.

In [ ]:
# ✅ Solution
image_dir = os.path.abspath("../image")
os.makedirs(image_dir, exist_ok=True)
print("Images:", image_dir)

💡 **Explanation**

From `notebooks/`, the relative path is `../image`, exactly like `../cache`.

## 6.3 · Chart 1: lap time evolution

> ✏️ **Exercise**
>
> Plot the lap times of both drivers over the race.
> - X axis: lap number. Y axis: time in seconds.
> - One line per driver.
> - A title, axis labels with units, a legend.
>
> Suggested matplotlib skeleton (unrelated example):
> ```python
> fig, ax = plt.subplots(figsize=(10, 5))
> ax.plot([1, 2, 3], [10, 12, 11], label="series A")
> ax.set_xlabel("...")
> ax.set_ylabel("...")
> ax.set_title("...")
> ax.legend()
> ```
>
> Bonus: `fastf1.plotting.get_driver_color` gives the official team colour of a driver.
>
> What can you read on the chart: tyre management? fuel burn? the moment the gap is built?

In [ ]:
# ✅ Solution
fig_laps, ax = plt.subplots(figsize=(10, 5))

for driver in DRIVERS:
    driver_laps = clean_laps[clean_laps["Driver"] == driver]
    colour = fastf1.plotting.get_driver_color(driver, session)
    ax.plot(driver_laps["LapNumber"], driver_laps["LapTime_s"],
            marker="o", markersize=4, linewidth=1, color=colour, label=driver)

ax.set_xlabel("Lap")
ax.set_ylabel("Lap time (s)")
ax.set_title(f"{session.event['EventName']} {YEAR}: lap times of {DRIVER_1} and {DRIVER_2}")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

💡 **Explanation**

Structure of a matplotlib figure: `fig` is the whole figure (the image), `ax` is the plotting area (the axes).
We draw on `ax`, and export `fig`.

- The two curves overlap almost all the way. In Monaco, overtaking is nearly impossible:
  Piastri stays behind Leclerc, and both manage their tyres by driving slowly early in the race
  (84 s on lap 3, 78.5 s around lap 20). Lap times then drop steadily as the car burns fuel.
- Between laps 66 and 72, Leclerc clearly speeds up (down to 75.2 s) while Piastri stays around 77 s.
  This is where most of the final gap is built. Leclerc's last lap, slower (78.6 s), is a celebration lap.

## 6.4 · Chart 2: lap time distribution

> ✏️ **Exercise**
>
> Draw a **boxplot** of the representative lap times of each driver. Look for `ax.boxplot` in the matplotlib documentation.
>
> 1. What does the line in the middle of the box represent?
> 2. What does the height of the box tell you?
> 3. Make sure the median line stays visible on top of a coloured box.

In [ ]:
# ✅ Solution
fig_distribution, ax = plt.subplots(figsize=(6, 5))

data = [clean_laps.loc[clean_laps["Driver"] == driver, "LapTime_s"] for driver in DRIVERS]
boxes = ax.boxplot(data, tick_labels=DRIVERS, patch_artist=True, medianprops={"color": "black"})

for box, driver in zip(boxes["boxes"], DRIVERS):
    box.set_facecolor(fastf1.plotting.get_driver_color(driver, session))
    box.set_alpha(0.6)

ax.set_ylabel("Lap time (s)")
ax.set_title("Distribution of representative lap times")
ax.grid(axis="y", alpha=0.3)
plt.show()

💡 **Explanation**

1. The middle line is the **median**.
2. The box contains half of the laps (from the 25th to the 75th percentile): the smaller it is, the more **consistent** the driver.
   Isolated points are unusual laps.
3. By default the median line is orange, which disappears on Piastri's orange box: `medianprops={"color": "black"}` fixes it.

Both medians sit at about 78.5 s. Leclerc's box reaches lower because of his fast laps at the end of the race.

## 6.5 · Chart 3: stints by tyre compound

> ✏️ **Exercise**
>
> Redo chart 1 with **one chart per driver**, side by side, and colour each point by the tyre compound used (`Compound`).
>
> Hints: `plt.subplots(1, 2, sharey=True)`, `groupby("Compound")`, `ax.scatter`,
> and `fastf1.plotting.get_compound_color` for the official tyre colours.
>
> Careful: the hard tyre colour is almost white. How can you keep the points visible on a white background?

In [ ]:
# ✅ Solution
fig_tyres, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, driver in zip(axes, DRIVERS):
    driver_laps = clean_laps[clean_laps["Driver"] == driver]
    for compound, compound_laps in driver_laps.groupby("Compound"):
        ax.scatter(compound_laps["LapNumber"], compound_laps["LapTime_s"], s=20,
                   color=fastf1.plotting.get_compound_color(compound, session),
                   edgecolors="black", linewidths=0.5, label=compound)
    ax.set_title(driver)
    ax.set_xlabel("Lap")
    ax.legend(title="Tyre")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Lap time (s)")
fig_tyres.suptitle("Lap times by tyre compound")
plt.show()

💡 **Explanation**

A black outline (`edgecolors="black"`) keeps the white hard-tyre points visible. `sharey=True` gives both charts
the same Y axis, so they can be compared.

Both drivers ran **the whole race on hard tyres**. They started on mediums but changed tyres during the
first-lap red flag, which spared them a pit stop. This chart will be more interesting on a race with several stints
(try changing `GRAND_PRIX`).

## 6.6 · Exporting

> ✏️ **Exercise**
>
> Save the three charts as PNG files in the folder prepared in 6.2.
> Look at the `dpi` and `bbox_inches` parameters of `savefig`: what do they change?

In [ ]:
# ✅ Solution
figures = {
    "lap_times.png": fig_laps,
    "lap_time_distribution.png": fig_distribution,
    "lap_times_by_tyre.png": fig_tyres,
}

for name, figure in figures.items():
    path = os.path.join(image_dir, name)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    print("Exported:", path)

💡 **Explanation**

`dpi=150` gives a sharp image, `bbox_inches="tight"` removes unnecessary white margins.
`os.path.join` builds the path with the right separator on every operating system.